In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.0 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login("-----------put your hf token here---------")

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train.jsonl to train (1).jsonl
Saving val.jsonl to val (1).jsonl


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "validation": "val.jsonl"
})

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # IMPORTANT

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # important for LLaMA-based models
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Print trainable params (for report)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
def format_example(example):
    return {
        "text": f"### Instruction:\n{example['input']}\n\n### Response:\n{example['output']}"
    }

dataset = dataset.map(format_example)

In [ ]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/932 [00:00<?, ? examples/s]

Map:   0%|          | 0/116 [00:00<?, ? examples/s]

In [ ]:
dataset.set_format(type="torch")

In [ ]:
print(dataset["train"][0])

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"]
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.143081,0.155061
2,0.149906,0.148948


TrainOutput(global_step=1864, training_loss=0.24032179592377126, metrics={'train_runtime': 629.1035, 'train_samples_per_second': 2.963, 'train_steps_per_second': 2.963, 'total_flos': 5930283090051072.0, 'train_loss': 0.24032179592377126, 'epoch': 2.0})

In [ ]:
model.save_pretrained("fine_tuned_model")
tokenizer.save_pretrained("fine_tuned_model")

('fine_tuned_model/tokenizer_config.json',
 'fine_tuned_model/chat_template.jinja',
 'fine_tuned_model/tokenizer.json')

In [ ]:
import os
print(os.listdir())

['.config', 'results', 'train.jsonl', 'val.jsonl', 'fine_tuned_model', '.ipynb_checkpoints', 'sample_data']


In [ ]:
!zip -r model.zip fine_tuned_model

  adding: fine_tuned_model/ (stored 0%)
  adding: fine_tuned_model/tokenizer_config.json (deflated 46%)
  adding: fine_tuned_model/tokenizer.json (deflated 85%)
  adding: fine_tuned_model/adapter_config.json (deflated 57%)
  adding: fine_tuned_model/adapter_model.safetensors (deflated 8%)
  adding: fine_tuned_model/chat_template.jinja (deflated 60%)
  adding: fine_tuned_model/README.md (deflated 66%)


In [ ]:
from google.colab import files
files.download("model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load fine-tuned model
model_path = "fine_tuned_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

# Set model to evaluation mode
model.eval()

# Test prompt (same format as training)
prompt = """### Instruction:
Write an SQL query to find the second highest salary from a table

### Response:
"""

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

# Decode result
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("🔹 MODEL OUTPUT:\n")
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

🔹 MODEL OUTPUT:

### Instruction:
Write an SQL query to find the second highest salary from a table

### Response:
SELECT name, salary
FROM employees
ORDER BY salary DESC
LIMIT 1;
